# 🎙️ Multilingual Audio RAG - Colab Edition

**GPU-accelerated RAG for English + Hindi transcripts**

This notebook runs entirely in Google Colab with T4/Tesla GPUs.

## Features:
- ✅ Auto-detects GPU availability
- ✅ Upload transcripts via Colab file browser
- ✅ GPU-accelerated embeddings
- ✅ Cloud LLM support (OpenAI/Anthropic/HuggingFace)
- ✅ Interactive step-by-step cells
- ✅ Persistent storage via Google Drive

## Instructions:
1. Open this notebook in Google Colab
2. Go to **Runtime > Change runtime type** and select **GPU**
3. Run each cell sequentially
4. Upload your transcript files when prompted
5. Ask questions in English or Hindi!

In [ ]:
# @title Step 1: Mount Google Drive and Setup
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

In [ ]:
# @title Step 2: Install Dependencies
!pip install -q sentence-transformers chromadb torch transformers accelerate bitsandbytes
!pip install -q openai anthropic huggingface-hub langchain-text-splitters langdetect tiktoken
!pip install -q ipywidgets

print("✅ Dependencies installed")

In [ ]:
# @title Step 2b: Prepare Project Files in Colab
import os
import sys
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/multilingual-audio-rag-colab")
REPO_URL = "https://github.com/amollate/multilingual-audio-rag-colab.git"

# Always work from /content to avoid broken cwd issues
os.chdir("/content")

def run(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, cwd=cwd or "/content", check=False, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Command failed: {cmd}")
        print(result.stderr)
    return result

if PROJECT_DIR.exists():
    print("📂 Repo exists, refreshing...")
    run("git pull", cwd=str(PROJECT_DIR))
else:
    print("📥 Cloning repository into /content...")
    run(f"git clone {REPO_URL} {PROJECT_DIR}")

if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)
    sys.path.append(str(PROJECT_DIR))
    print(f"✅ Project ready at: {PROJECT_DIR}")
    print(f"   Contents: {os.listdir(PROJECT_DIR)}")
else:
    print("❌ Git clone failed. Falling back to writing source files directly...")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.append(str(PROJECT_DIR))
    print("⚠️  Running in fallback mode. Some features may be limited.")


In [ ]:
# @title Step 3: GPU and Environment Check
import torch
import sys
from pathlib import Path

# Check GPU
gpu_available = torch.cuda.is_available()
print(f"\n{'='*50}")
print(f"GPU Available: {gpu_available}")
if gpu_available:
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU detected. Running on CPU will be slower.")
print(f"{'='*50}\n")

# Check Python version
print(f"Python Version: {sys.version}")

# Setup directories
DRIVE_PATH = "/content/drive/MyDrive/multilingual-audio-rag"
DATA_DIR = Path(DRIVE_PATH) / "data"
TRANSCRIPT_DIR = DATA_DIR / "transcripts"
PROCESSED_DIR = DATA_DIR / "processed"
VECTOR_DB_PATH = PROCESSED_DIR / "chroma_db"

for dir_path in [DATA_DIR, TRANSCRIPT_DIR, PROCESSED_DIR, VECTOR_DB_PATH]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Directories created at: {DRIVE_PATH}")
print(f"   - Transcripts: {TRANSCRIPT_DIR}")
print(f"   - Vector DB: {VECTOR_DB_PATH}")

In [ ]:
# @title Step 4: Upload Transcript Files
from google.colab import files
import shutil

print("📤 Upload your transcript files (.txt or .json)")
print("   You can select multiple files at once\n")

uploaded = files.upload()

# Move uploaded files to transcript directory
for filename in uploaded.keys():
    src = f"/content/{filename}"
    dst = TRANSCRIPT_DIR / filename
    shutil.move(src, dst)
    print(f"   ✅ Moved {filename} -> {dst}")

print(f"\n📊 Total transcripts uploaded: {len(uploaded)}")

In [ ]:
# @title Step 5: Configure LLM Provider
import os
from google.colab import widgets

# Get API keys from user
print("\n🔑 LLM Provider Configuration")
print("Choose your LLM provider and enter API key when prompted\n")

# For simplicity, we'll use OpenAI as default
# You can modify this to support other providers
OPENAI_API_KEY = "" # @param {type:"string"}

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("✅ OpenAI API key set")
else:
    print("⚠️  No API key provided. Using HuggingFace models (slower)")

LLM_MODEL = "gpt-4o-mini" # @param ["gpt-4o-mini", "gpt-3.5-turbo", "claude-3-5-haiku-20240620"]
print(f"\n🤖 Selected LLM: {LLM_MODEL}")

In [ ]:
# @title Step 6: Initialize RAG Pipeline
import sys
sys.path.append('/content/drive/MyDrive/multilingual-audio-rag-colab')

from src.config import ColabConfig
from src.pipeline.rag_pipeline import RAGPipeline

# Create config with auto-detection
config = ColabConfig(drive_mount_path="/content/drive/MyDrive/multilingual-audio-rag")

# Override with user settings
if OPENAI_API_KEY:
    config.llm_provider = "openai"
    config.openai_api_key = OPENAI_API_KEY
    config.llm_model = LLM_MODEL
else:
    config.llm_provider = "huggingface"
    config.llm_model = "HuggingFaceH4/zephyr-7b-beta"

# Setup directories
config.setup_directories()

# Initialize pipeline
print("\n🚀 Initializing RAG Pipeline...")
pipeline = RAGPipeline(config)

# Show status
status = pipeline.get_status()
print("\n📊 System Status:")
for key, value in status.items():
    print(f"   {key}: {value}")

In [ ]:
# @title Step 7: Ingest Transcripts
print("\n📚 Starting transcript ingestion...")
print("   This will:")
print("   1. Parse all uploaded transcripts")
print("   2. Split into chunks (~500 chars each)")
print("   3. Create embeddings using GPU")
print("   4. Store in ChromaDB\n")

# Run ingestion
result = pipeline.ingest_transcripts(str(TRANSCRIPT_DIR))

print("\n✅ Ingestion Complete!")
print(f"   Status: {result['status']}")
print(f"   Transcripts processed: {result.get('transcripts_processed', 0)}")
print(f"   Chunks created: {result.get('chunks_created', 0)}")
print(f"   Total documents in DB: {result.get('total_documents', 0)}")

In [ ]:
# @title Step 8: Query Your Transcripts!
import ipywidgets as widgets
from IPython.display import display, HTML

print("\n💬 Ask questions about your transcripts!")
print("   Try questions in English or Hindi\n")

# Create input widgets
question_input = widgets.Text(
    placeholder='Enter your question here...',
    description='Question:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

language_dropdown = widgets.Dropdown(
    options=['en', 'hi'],
    value='en',
    description='Language:',
    style={'description_width': 'initial'}
)

ask_button = widgets.Button(
    description='Ask',
    button_style='success',
    tooltip='Submit question'
)

output_area = widgets.Output()

def on_ask_button_clicked(b):
    question = question_input.value.strip()
    if not question:
        with output_area:
            output_area.clear_output()
            print("⚠️  Please enter a question")
        return
    
    language = language_dropdown.value
    
    with output_area:
        output_area.clear_output()
        print(f"\n🔍 Question: {question}")
        print(f"🌐 Language: {language}")
        print("\n⏳ Searching and generating answer...")
        
        try:
            result = pipeline.query(question, language=language)
            
            print(f"\n📝 Answer:")
            print(f"{result['answer']}")
            
            if result.get('sources'):
                print(f"\n📎 Sources ({result['retrieved_chunks']} chunks):")
                for source in result['sources'][:3]:
                    print(f"   - {source}")
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")

ask_button.on_click(on_ask_button_clicked)

# Display widgets
display(widgets.HBox([question_input, language_dropdown, ask_button]))
display(output_area)

In [ ]:
# @title Step 9: Explore Your Data (Optional)
# Show statistics about ingested data

import pandas as pd

# Get all documents
docs = pipeline.vector_store.list_documents(limit=1000)

if docs:
    # Create DataFrame
    df = pd.DataFrame([
        {
            "id": d["id"],
            "source": d["metadata"].get("source", "unknown"),
            "language": d["metadata"].get("language", "unknown"),
            "chunk_size": d["metadata"].get("chunk_size", 0),
            "text_preview": d["text"][:100] + "..." if len(d["text"]) > 100 else d["text"]
        }
        for d in docs
    ])
    
    print(f"\n📊 Total chunks: {len(df)}")
    print(f"\n📈 By Language:")
    print(df["language"].value_counts())
    print(f"\n📁 By Source:")
    print(df["source"].value_counts().head(10))
    
    # Display sample
    print(f"\n📄 Sample chunks:")
    display(df.head(5))
else:
    print("No documents in database")

## 🎯 Quick Tips

1. **GPU Detection**: The notebook auto-detects your GPU. If no GPU is found, it falls back to CPU (slower).
2. **Persistence**: All data is saved to Google Drive, so it persists across sessions.
3. **Model Selection**: Choose the fastest model you have API access to for best experience.
4. **Transcript Format**: Supports `.txt` (line-numbered or plain) and `.json` formats.
5. **Languages**: Works with English, Hindi, and mixed content.

## 🔧 Troubleshooting

- **Out of Memory**: Reduce `chunk_size` in config or use smaller embedding model
- **Slow Responses**: Use GPT-4o-mini or Claude Haiku for fastest responses
- **Upload Issues**: Make sure files are .txt or .json format, UTF-8 encoded

## 📝 Notes

- First run will take longer as it downloads models
- Subsequent runs will be faster due to caching
- Vector DB is persisted in Google Drive
- You can stop the runtime when not in use